In [111]:
import os
import json
import random
import pickle as pkl
import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedKFold
from sklearn import metrics
from fastnode2vec import Node2Vec, Graph
from gensim.models import Word2Vec
from datasets import load_metric
from utils import get_texts_for_ptype

import xgboost as xgb
import numpy as np
import scipy as sp
from sklearn.linear_model import SGDClassifier
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer
from custom_models.custom_model import BatchVectorizerXGBoostConcat, BatchVectorizerXGBoost

from utils import get_texts_for_ptype_pmid, get_embeddings, get_texts_citations_labels

In [2]:
study_design = 'is_rct'

In [3]:
neg_to_pos_ratio = 1

In [4]:
neg_to_pos_ratio = 1
results_node2vec_gbdt_file = "results_node2vec_gbdt_sample.json"
results_texts_gbdt_file = "results_titles_abstracts_gbdt_features1000.json"
MODEL_DIR = "../models"
EMBEDDING_FILENAME = os.path.join(MODEL_DIR, 'node_model.pkl')
PMIDS = '../data/pmids.pkl'

In [5]:
with open(results_node2vec_gbdt_file, "r") as f:
    results_node2vec_gbdt_list = json.load(f)

In [6]:
with open(results_texts_gbdt_file, "r") as f:
    results_texts_gbdt_list = json.load(f)

In [8]:
with open(PMIDS, 'rb') as f:
    pmids = pkl.load(f)

In [10]:
with open("../data/pmid_to_openAlex.json") as f:
    pmid_to_openAlex = json.load(f)

In [13]:
with open("../data/graph_lst.pkl", "rb") as f:
    graph_lst = pkl.load(f)

In [14]:
graph = Graph(graph_lst, directed=False, weighted=False)
node2vec = Node2Vec(graph, dim=10, walk_length=100, context=10, p=2.0, q=0.5, workers=5)  # Use temp_folder for big graphs
model = node2vec.wv.load_word2vec_format(EMBEDDING_FILENAME)

Reading graph: 100%|█████████████████████████████████████████| 39051/39051 [00:00<00:00, 1506699.22it/s]


In [106]:
ensemble_results_dict = {}
ensemble_results_file = "results_ensemble_text_node2vec_gbdt.json"

for results_node2vec, results_texts in zip(results_node2vec_gbdt_list, results_texts_gbdt_list):
    assert results_node2vec['ptype'] == results_texts['ptype']
    ptype = results_node2vec['ptype']
    ensemble_results_dict[ptype] = {}
    
    pos_train_embeddings, neg_train_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='train')
    pos_val_embeddings, neg_val_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='val')
    pos_test_embeddings, neg_test_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='test')
        
    pos_train_pmids = list(pos_train_embeddings.keys())
    neg_train_pmids = list(neg_train_embeddings.keys())
    pos_val_pmids = list(pos_val_embeddings.keys())
    neg_val_pmids = list(neg_val_embeddings.keys())
    pos_test_pmids = list(pos_test_embeddings.keys())
    neg_test_pmids = list(neg_test_embeddings.keys())
    pmids_to_use = {
        "pos_train": pos_train_pmids,
        "neg_train": neg_train_pmids,
        "pos_val": pos_val_pmids,
        "neg_val": neg_val_pmids,
        "pos_test": pos_test_pmids,
        "neg_test": neg_test_pmids
    }
    
    train_set, val_set, test_set = get_texts_for_ptype_pmid(ptype, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)
        
    train_set_concat = {}
    for pmid in pos_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = pos_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 1

    for pmid in neg_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = neg_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 0

    val_set_concat = {}
    for pmid in pos_val_pmids:
        val_set_concat[pmid] = {}
        val_set_concat[pmid]["X_graph"] = pos_val_embeddings[pmid]
        val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
        val_set_concat[pmid]["y"] = 1

    for pmid in neg_val_pmids:
        val_set_concat[pmid] = {}
        val_set_concat[pmid]["X_graph"] = neg_val_embeddings[pmid]
        val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
        val_set_concat[pmid]["y"] = 0

    test_set_concat = {}
    for pmid in pos_test_pmids:
        test_set_concat[pmid] = {} 
        test_set_concat[pmid]["X_graph"] = pos_test_embeddings[pmid]
        test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
        test_set_concat[pmid]["y"] = 1

    for pmid in neg_test_pmids:
        test_set_concat[pmid] = {}
        test_set_concat[pmid]["X_graph"] = neg_test_embeddings[pmid]
        test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
        test_set_concat[pmid]["y"] = 0
        
    train_df = get_texts_citations_labels(train_set_concat)
    train_df = train_df.sample(frac=1)
        
    val_df = get_texts_citations_labels(val_set_concat)
    val_df = val_df.sample(frac=1)
        
    test_df = get_texts_citations_labels(test_set_concat)
    test_df = test_df.sample(frac=1)
    
    bvc_node2vec = xgb.XGBClassifier(
        objective="binary:logistic",
        max_depth=18,
        gamma=results_node2vec["gamma"],
        reg_alpha=results_node2vec["reg_alpha"],
        reg_lambda=results_node2vec["reg_lambda"],
        colsample_bytree=results_node2vec["colsample_bytree"],
        min_child_weight=results_node2vec["min_child_weight"],
        n_estimators=180,
        seed=0,
    )
    
    bvc_node2vec.fit(np.vstack(train_df["node_embeds"]), np.array(train_df["labels"]))
    
    val_node2vec_probs = bvc_node2vec.predict_proba(np.vstack(val_df["node_embeds"]))
    val_node2vec_probs = np.array([pos for neg, pos in val_node2vec_probs])
    
    test_node2vec_probs = bvc_node2vec.predict_proba(np.vstack(test_df["node_embeds"]))
    test_node2vec_probs = np.array([pos for neg, pos in test_node2vec_probs])
    
    bvc_text = BatchVectorizerXGBoost(
        objective="binary:logistic",
        max_depth=18,
        gamma=results_texts["gamma"],
        reg_alpha=results_texts["reg_alpha"],
        reg_lambda=results_texts["reg_lambda"],
        colsample_bytree=results_texts["colsample_bytree"],
        min_child_weight=results_texts["min_child_weight"],
        n_estimators=180,
        seed=0,
    )
    
    bvc_text.fit(train_df["texts"], np.array(train_df["labels"]))
    val_text_probs = bvc_text.predict_proba(val_df["texts"])
    val_text_probs = np.array([pos for neg, pos in val_text_probs])
    
    test_text_probs = bvc_text.predict_proba(test_df["texts"])
    test_text_probs = np.array([pos for neg, pos in test_text_probs])
    
    weights = {
        "even": 0.5,
        "proportional": np.vstack(train_df['node_embeds']).shape[1]/1000,
        "learnt": max(min(1, np.sum(val_df['labels']-val_text_probs)/np.sum(val_node2vec_probs-val_text_probs)), 0)
    }
    
    for name, weight in weights.items():
        test_ensemble_probs = weight*test_node2vec_probs + (1-weight)*test_text_probs
        y_pred_test = np.round(test_ensemble_probs)
        y_test = test_df["labels"]
        
        recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
        spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
        precision_test = metrics.precision_score(y_test, y_pred_test)
        f1_test = metrics.f1_score(y_test, y_pred_test)
        accuracy_test = metrics.accuracy_score(y_test, y_pred_test)
        
        ensemble_results_dict[ptype][name] = {
            "weight": weight,
            "accuracy_test": accuracy_test,
            "recall_test": recall_test,
            "spec_test": spec_test,
            "precision_test_even": precision_test,
            "f1_test_even": f1_test,s
            "accuracy_test": accuracy_test,
        }
    with open(ensemble_results_file, "w") as f:
        json.dump(ensemble_results_dict, f)
    

100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 561411.32it/s]
19it [00:26,  1.41s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:12,  1.46s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:02,  1.43s/it]
57it [01:32,  1.63s/it]
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [

[20:28:22] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[20:28:24] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 396558.87it/s]
19it [00:28,  1.51s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:15,  1.48s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:05,  1.46s/it]
57it [01:32,  1.62s/it]
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [

...

[20:40:10] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 393886.84it/s]
19it [00:28,  1.52s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:15,  1.48s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:05,  1.46s/it]
57it [01:32,  1.62s/it]
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [

[20:44:11] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[20:44:13] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [122]:
ensemble_crossval_results_dict = {}
ensemble_crossval_results_file = "results_crossval_ensemble_text_node2vec_gbdt.json"

for results_node2vec, results_texts in zip(results_node2vec_gbdt_list, results_texts_gbdt_list):
    assert results_node2vec['ptype'] == results_texts['ptype']
    ptype = results_node2vec['ptype']
    ensemble_crossval_results_dict[ptype] = {}
    
    pos_train_embeddings, neg_train_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='train')
    pos_val_embeddings, neg_val_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='val')
    pos_test_embeddings, neg_test_embeddings = get_embeddings(pmids, pmid_to_openAlex, ptype, model, split='test')
        
    pos_train_pmids = list(pos_train_embeddings.keys())
    neg_train_pmids = list(neg_train_embeddings.keys())
    pos_val_pmids = list(pos_val_embeddings.keys())
    neg_val_pmids = list(neg_val_embeddings.keys())
    pos_test_pmids = list(pos_test_embeddings.keys())
    neg_test_pmids = list(neg_test_embeddings.keys())
    pmids_to_use = {
        "pos_train": pos_train_pmids,
        "neg_train": neg_train_pmids,
        "pos_val": pos_val_pmids,
        "neg_val": neg_val_pmids,
        "pos_test": pos_test_pmids,
        "neg_test": neg_test_pmids
    }
    
    train_set, val_set, test_set = get_texts_for_ptype_pmid(ptype, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)
        
    train_set_concat = {}
    for pmid in pos_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = pos_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 1

    for pmid in neg_train_pmids:
        train_set_concat[pmid] = {}
        train_set_concat[pmid]["X_graph"] = neg_train_embeddings[pmid]
        train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
        train_set_concat[pmid]["y"] = 0

#     val_set_concat = {}
#     for pmid in pos_val_pmids:
#         val_set_concat[pmid] = {}
#         val_set_concat[pmid]["X_graph"] = pos_val_embeddings[pmid]
#         val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
#         val_set_concat[pmid]["y"] = 1

#     for pmid in neg_val_pmids:
#         val_set_concat[pmid] = {}
#         val_set_concat[pmid]["X_graph"] = neg_val_embeddings[pmid]
#         val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
#         val_set_concat[pmid]["y"] = 0

#     test_set_concat = {}
#     for pmid in pos_test_pmids:
#         test_set_concat[pmid] = {} 
#         test_set_concat[pmid]["X_graph"] = pos_test_embeddings[pmid]
#         test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
#         test_set_concat[pmid]["y"] = 1

#     for pmid in neg_test_pmids:
#         test_set_concat[pmid] = {}
#         test_set_concat[pmid]["X_graph"] = neg_test_embeddings[pmid]
#         test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
#         test_set_concat[pmid]["y"] = 0
        
    train_df = get_texts_citations_labels(train_set_concat)
    train_df = train_df.sample(frac=1)
        
#     val_df = get_texts_citations_labels(val_set_concat)
#     val_df = val_df.sample(frac=1)
        
#     test_df = get_texts_citations_labels(test_set_concat)
#     test_df = test_df.sample(frac=1)
    
    
    
    
#     test_node2vec_probs = bvc_node2vec.predict_proba(np.vstack(test_df["node_embeds"]))
#     test_node2vec_probs = np.array([pos for neg, pos in test_node2vec_probs])
    
   
#     val_text_probs = bvc_text.predict_proba(val_df["texts"])
#     val_text_probs = np.array([pos for neg, pos in val_text_probs])
    
#     test_text_probs = bvc_text.predict_proba(test_df["texts"])
#     test_text_probs = np.array([pos for neg, pos in test_text_probs])
    
#     weights = {
#         "even": 0.5,
#         "proportional": np.vstack(train_df['node_embeds']).shape[1]/1000,
#         "learnt": max(min(1, np.sum(val_df['labels']-val_text_probs)/np.sum(val_node2vec_probs-val_text_probs)), 0)
#     }
    random_state = 12883823
    rkf = RepeatedKFold(n_splits=10, n_repeats=1, random_state=random_state)
    score_dict = {
        "even": {
            "accuracy_test": [],
            "recall_test": [],
            "spec_test": [],
            "precision_test_even": [],
            "f1_test_even": [],
        },
        "proportional": {
            "accuracy_test": [],
            "recall_test": [],
            "spec_test": [],
            "precision_test_even": [],
            "f1_test_even": [],
        },
        "learnt" : {
            "accuracy_test": [],
            "recall_test": [],
            "spec_test": [],
            "precision_test_even": [],
            "f1_test_even": [],
        }
    }
    
    for fold, (train_idx, valid_idx) in enumerate(tqdm(rkf.split(train_df.index))):
        train_subset = train_df.loc[train_idx]
        val_subset = train_df.loc[valid_idx]
        
        
        
        bvc_node2vec = xgb.XGBClassifier(
                objective="binary:logistic",
                max_depth=18,
                gamma=results_node2vec["gamma"],
                reg_alpha=results_node2vec["reg_alpha"],
                reg_lambda=results_node2vec["reg_lambda"],
                colsample_bytree=results_node2vec["colsample_bytree"],
                min_child_weight=results_node2vec["min_child_weight"],
                n_estimators=180,
                seed=0,
        )

        bvc_node2vec.fit(np.vstack(train_subset["node_embeds"]), np.array(train_subset["labels"]))
        
        val_node2vec_probs = bvc_node2vec.predict_proba(np.vstack(val_subset["node_embeds"]))
        val_node2vec_probs = np.array([pos for neg, pos in val_node2vec_probs])
            
            
        bvc_text = BatchVectorizerXGBoost(
            objective="binary:logistic",
            max_depth=18,
            gamma=results_texts["gamma"],
            reg_alpha=results_texts["reg_alpha"],
            reg_lambda=results_texts["reg_lambda"],
            colsample_bytree=results_texts["colsample_bytree"],
            min_child_weight=results_texts["min_child_weight"],
            n_estimators=180,
            seed=0,
        )
        bvc_text.fit(train_subset["texts"], np.array(train_subset["labels"]))
        
        val_text_probs = bvc_text.predict_proba(val_subset["texts"])
        val_text_probs = np.array([pos for neg, pos in val_text_probs])
        
        for name in ensemble_results_dict[ptype]:
            weight = ensemble_results_dict[ptype][name]['weight']
            
            test_ensemble_probs = weight*val_node2vec_probs + (1-weight)*val_text_probs
            


            y_pred_test = np.round(test_ensemble_probs)
            y_test = val_subset["labels"]

            recall_test = metrics.recall_score(y_test, y_pred_test, pos_label=1)
            spec_test = metrics.recall_score(y_test, y_pred_test, pos_label=0)
            precision_test = metrics.precision_score(y_test, y_pred_test)
            f1_test = metrics.f1_score(y_test, y_pred_test)
            accuracy_test = metrics.accuracy_score(y_test, y_pred_test)
            
            score_dict[name]["accuracy_test"].append(accuracy_test)
            score_dict[name]["recall_test"].append(recall_test)
            score_dict[name]["spec_test"].append(spec_test)
            score_dict[name]["precision_test_even"].append(precision_test)
            score_dict[name]["f1_test_even"].append(f1_test)
            score_dict[name]["weight"] = weight

        
    ensemble_crossval_results_dict[ptype] = score_dict
        
    with open(ensemble_crossval_results_file, "w") as f:
        json.dump(ensemble_crossval_results_dict, f)
    

100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 533033.07it/s]
19it [00:28,  1.52s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:15,  1.48s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:07,  1.49s/it]
57it [01:36,  1.70s/it]
0it [00:00, ?it/s]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.

[14:24:19] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:24:21] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


1it [00:08,  8.43s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


...

9it [00:43,  4.77s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:49:53] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:49:55] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


10it [00:47,  4.78s/it]


In [121]:
len(y_pred_test)

6979

In [120]:
len(y_test)

744

In [101]:
bvc_text.predict(test_df["texts"])

array([1, 0, 0, ..., 0, 1, 0])

In [104]:
np.round(test_text_probs)

array([0., 1., 1., ..., 1., 0., 1.], dtype=float32)

In [91]:
max(min(1, np.sum(val_df['labels']-val_text_probs)/np.sum(val_node2vec_probs-val_text_probs)), 0)

1

In [96]:
y_pred_test

array([1., 0., 1., ..., 0., 0., 0.], dtype=float32)

In [97]:
y_test

1741    1
5564    0
4814    0
3179    1
4391    0
       ..
6052    0
3272    1
3369    0
1730    1
6774    0
Name: labels, Length: 6959, dtype: int64

In [100]:
test_df['labels']

1741    1
5564    0
4814    0
3179    1
4391    0
       ..
6052    0
3272    1
3369    0
1730    1
6774    0
Name: labels, Length: 6959, dtype: int64

In [35]:
X_train, X_val, X_test, y_train, y_val, y_test = get_texts_for_ptype(ptype, inputs="titles_abstracts", neg_to_pos_ratio=neg_to_pos_ratio)

19it [00:27,  1.46s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:13,  1.48s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:02,  1.42s/it]
57it [01:32,  1.61s/it]


In [31]:
bvc = BatchVectorizerXGBoost(
        objective="binary:logistic",
        max_depth=18,
        gamma=results_node2vec["gamma"],
        reg_alpha=results_node2vec["reg_alpha"],
        reg_lambda=results_node2vec["reg_lambda"],
        colsample_bytree=results_node2vec["colsample_bytree"],
        min_child_weight=results_node2vec["min_child_weight"],
        n_estimators=180,
        seed=0,
    )

In [38]:
bvc.fit(X_train, y_train)

/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[13:06:59] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [40]:
pred_probs = bvc.predict_proba(X_train)

In [54]:
scores = [pos for pos, neg in pred_probs]
scores = np.array(scores)

In [56]:
scores = np.round(scores)
scores

array([1., 1., 1., ..., 1., 0., 1.], dtype=float32)

In [20]:
pos_train_embeddings, neg_train_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='train')

100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 518007.16it/s]


In [21]:
pos_val_embeddings, neg_val_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='val')

100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 512782.44it/s]


In [22]:
pos_test_embeddings, neg_test_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='test')

100%|███████████████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 534510.51it/s]


In [23]:
pos_train_pmids = list(pos_train_embeddings.keys())
neg_train_pmids = list(neg_train_embeddings.keys())
pos_val_pmids = list(pos_val_embeddings.keys())
neg_val_pmids = list(neg_val_embeddings.keys())
pos_test_pmids = list(pos_test_embeddings.keys())
neg_test_pmids = list(neg_test_embeddings.keys())

In [24]:
pmids_to_use = {
    "pos_train": pos_train_pmids,
    "neg_train": neg_train_pmids,
    "pos_val": pos_val_pmids,
    "neg_val": neg_val_pmids,
    "pos_test": pos_test_pmids,
    "neg_test": neg_test_pmids
}

In [25]:
train_set, val_set, test_set = get_texts_for_ptype_pmid(study_design, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)

19it [00:26,  1.44s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:12,  1.43s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [02:02,  1.42s/it]
57it [01:32,  1.62s/it]


In [18]:
train_set_concat = {}
for pmid in pos_train_pmids:
    train_set_concat[pmid] = {}
    train_set_concat[pmid]["X_graph"] = pos_train_embeddings[pmid]
    train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
    train_set_concat[pmid]["y"] = 1

for pmid in neg_train_pmids:
    train_set_concat[pmid] = {}
    train_set_concat[pmid]["X_graph"] = neg_train_embeddings[pmid]
    train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
    train_set_concat[pmid]["y"] = 0

val_set_concat = {}
for pmid in pos_val_pmids:
    val_set_concat[pmid] = {}
    val_set_concat[pmid]["X_graph"] = pos_val_embeddings[pmid]
    val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
    val_set_concat[pmid]["y"] = 1

for pmid in neg_val_pmids:
    val_set_concat[pmid] = {}
    val_set_concat[pmid]["X_graph"] = neg_val_embeddings[pmid]
    val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
    val_set_concat[pmid]["y"] = 0

test_set_concat = {}
for pmid in pos_test_pmids:
    test_set_concat[pmid] = {} 
    test_set_concat[pmid]["X_graph"] = pos_test_embeddings[pmid]
    test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
    test_set_concat[pmid]["y"] = 1

for pmid in neg_test_pmids:
    test_set_concat[pmid] = {}
    test_set_concat[pmid]["X_graph"] = neg_test_embeddings[pmid]
    test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
    test_set_concat[pmid]["y"] = 0

In [51]:
train_df = get_texts_citations_labels(train_set_concat)
train_df = train_df.sample(frac=1)

NameError: name 'train_set_concat' is not defined

In [21]:
val_df = get_texts_citations_labels(val_set_concat)
val_df = val_df.sample(frac=1)

In [22]:
test_df = get_texts_citations_labels(test_set_concat)
test_df = test_df.sample(frac=1)

In [23]:
len(train_df)

7437

In [24]:
len(val_df)

7370

In [25]:
len(test_df)

7406

10